# Phase 6: TS1 Mechanism (~70 species)

Verifies the full TS1 tropospheric/stratospheric mechanism:
- Config-only switch from Chapman (no recompilation)
- O3, NO, NO2, CO show expected diurnal patterns
- No negative species concentrations
- Performance: < 2 sec/step

**Pre-requisite:** `data/jw_480km_ts1/output.nc`

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data" / "jw_480km_ts1"
OUTPUT = DATA_DIR / "output.nc"
assert OUTPUT.exists(), f"Run the Phase 6 test first — {OUTPUT} not found"

ds = nc.Dataset(OUTPUT)
lat = np.degrees(ds["latCell"][:])
lon = np.degrees(ds["lonCell"][:])
nTimes = ds.dimensions["Time"].size
nLevels = ds.dimensions["nVertLevels"].size
print(f"Grid: {ds.dimensions['nCells'].size} cells, {nLevels} levels, {nTimes} steps")

## 1. Key Species Diurnal Cycles

Domain-mean mixing ratios over time for O3, NO, NO2, CO at a
representative level.

In [ ]:
key_species = ["O3", "NO", "NO2", "CO"]
lev = 6  # ~500 hPa

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, sp in zip(axes.flat, key_species):
    if sp in ds.variables:
        mean = np.array([ds[sp][t, :, lev].mean() for t in range(nTimes)])
        vmin = np.array([ds[sp][t, :, lev].min() for t in range(nTimes)])
        vmax = np.array([ds[sp][t, :, lev].max() for t in range(nTimes)])
        ax.fill_between(range(nTimes), vmin, vmax, alpha=0.2)
        ax.plot(range(nTimes), mean, "o-", markersize=3)
        ax.set_title(sp)
        ax.set_xlabel("Time step")
        ax.set_ylabel("mol/mol")
    else:
        ax.text(0.5, 0.5, f"{sp} not found", ha="center", va="center",
                transform=ax.transAxes)

plt.suptitle("Key Species Diurnal Cycles at ~500 hPa")
plt.tight_layout()
plt.show()

## 2. O3 Spatial Distribution

Map of O3 at ~500 hPa showing dayside/nightside contrast.

In [ ]:
if "O3" in ds.variables:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, t, label in zip(axes, [0, -1], ["Initial", "Final"]):
        vals = ds["O3"][t, :, lev]
        sc = ax.scatter(lon, lat, c=vals * 1e6, s=4, cmap="YlGn")
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.set_title(f"O3 at ~500 hPa — {label}")
        plt.colorbar(sc, ax=ax, label="ppmv")
    plt.tight_layout()
    plt.show()

## 3. Negative Species Check

In [ ]:
chem_vars = [v for v in ds.variables
             if len(ds[v].dimensions) == 3
             and v not in {"pressure_base", "pressure_p", "theta",
                          "uReconstructZonal", "uReconstructMeridional"}]

print(f"Checking {len(chem_vars)} species for negative values...")
negatives = []
for v in chem_vars:
    min_val = ds[v][:].min()
    if min_val < 0:
        negatives.append((v, min_val))

if negatives:
    print("\n  NEGATIVE VALUES FOUND:")
    for v, val in negatives:
        print(f"    {v}: min = {val:.3e}")
else:
    print("  All species non-negative — PASS")

ds.close()